#### Promptable explanations for domain tasks, supported by the XAI QB

In [1]:
# Import modules, custom functions, data
import sys; sys.path.append("../"); from initialise import *

In [2]:
df = pd.read_csv("data/dengue_features_train.csv").merge(pd.read_csv("data/dengue_labels_train.csv"))

In [16]:
# Initial inspection of data
df.head()

,city,year,weekofyear,week_start_date,ndvi_ne,ndvi_nw,ndvi_se,ndvi_sw,precipitation_amt_mm,reanalysis_air_temp_k,...,reanalysis_relative_humidity_percent,reanalysis_sat_precip_amt_mm,reanalysis_specific_humidity_g_per_kg,reanalysis_tdtr_k,station_avg_temp_c,station_diur_temp_rng_c,station_max_temp_c,station_min_temp_c,station_precip_mm,total_cases
0,sj,1990,18,1990-04-30,0.122600,0.103725,0.198483,0.177617,12.42,297.572857,...,73.365714,12.42,14.012857,2.628571,25.442857,6.900000,29.4,20.0,16.0,4
1,sj,1990,19,1990-05-07,0.169900,0.142175,0.162357,0.155486,22.82,298.211429,...,77.368571,22.82,15.372857,2.371429,26.714286,6.371429,31.7,22.2,8.6,5
2,sj,1990,20,1990-05-14,0.032250,0.172967,0.157200,0.170843,34.54,298.781429,...,82.052857,34.54,16.848571,2.300000,26.714286,6.485714,32.2,22.8,41.4,4
3,sj,1990,21,1990-05-21,0.128633,0.245067,0.227557,0.235886,15.36,298.987143,...,80.337143,15.36,16.672857,2.428571,27.471429,6.771429,33.3,23.3,4.0,3
4,sj,1990,22,1990-05-28,0.196200,0.262200,0.251200,0.247340,7.52,299.518571,...,80.460000,7.52,17.210000,3.014286,28.942857,9.371429,35.0,23.9,5.8,6


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1456 entries, 0 to 1455
Data columns (total 25 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   city                                   1456 non-null   object 
 1   year                                   1456 non-null   int64  
 2   weekofyear                             1456 non-null   int64  
 3   week_start_date                        1456 non-null   object 
 4   ndvi_ne                                1262 non-null   float64
 5   ndvi_nw                                1404 non-null   float64
 6   ndvi_se                                1434 non-null   float64
 7   ndvi_sw                                1434 non-null   float64
 8   precipitation_amt_mm                   1443 non-null   float64
 9   reanalysis_air_temp_k                  1446 non-null   float64
 10  reanalysis_avg_temp_k                  1446 non-null   float64
 11  rean

In [21]:
# For the purpose of eda we can drop `Response ID`
prettyDescribe(df)

,min,mean,max,std,25%,50%,75%
year,1990.00,2001.03,2010.00,5.41,1997.00,2002.00,2005.00
weekofyear,1.00,26.50,53.00,15.02,13.75,26.50,39.25
ndvi_ne,-0.41,0.14,0.51,0.14,0.04,0.13,0.25
ndvi_nw,-0.46,0.13,0.45,0.12,0.05,0.12,0.22
ndvi_se,-0.02,0.20,0.54,0.07,0.16,0.20,0.25
ndvi_sw,-0.06,0.20,0.55,0.08,0.14,0.19,0.25
precipitation_amt_mm,0.00,45.76,390.60,43.72,9.80,38.34,70.24
reanalysis_air_temp_k,294.64,298.70,302.20,1.36,297.66,298.65,299.83
reanalysis_avg_temp_k,294.89,299.23,302.93,1.26,298.26,299.29,300.21
reanalysis_dew_point_temp_k,289.64,295.25,298.45,1.53,294.12,295.64,296.46


In [6]:
# Inspect our target variable `full vaccine`
fig = (px
 .pie(df, names = "full vaccine")
 .update_traces(marker_line_width = 1, marker_line_color = "white")
 .update_layout(title = dict(text = "Full Vaccine", x = 0.5))
)

# config = {"staticPlot": True}

fig.show()

In [7]:
# , zmin = -1, zmax = 1
px.imshow(df.corr(), height = 800, width = 1000, color_continuous_scale = px.colors.sequential.Viridis).update_xaxes(tickangle = 45)

In [8]:
# Feature correlation with our response variable `full vaccine`
vaccine_corr = (df
 .corr()[["full vaccine"]]
 .drop("full vaccine", axis = 0)
 .sort_values(by = "full vaccine", ascending = False)
 .reset_index()
 )

vaccine_corr

,index,full vaccine
0,age,0.304831
1,education level,0.249108
2,income,0.246019
3,employment,0.151981
4,tradeoff range,0.093902
5,loss resilience,0.075932
6,insurance,0.073863
7,risk aversion,0.040842
8,reward aversion consistency,0.029897
9,reward tipping point,0.026291


In [9]:
# Inspect the 3 highest and lowest correlated features with `full vaccine`
for var in list(vaccine_corr["index"][:3]) + list(vaccine_corr["index"][-3:]):

    fig = make_subplots(rows = 1, cols = 2, subplot_titles = ["Histogram", "100% Stacked Bar Chart"])
    
    for col in [1, 2]:

        tickvals, nbins = (df[var].unique(), None) if (df[var].dtype == "int64") & (len(df[var].unique()) < 10) else (None, 10)
        df_temp = df.value_counts(["full vaccine", var]).reset_index(name = "freq").sort_values(["full vaccine", var]).reset_index(drop = True)
        
        text_auto = False if col == 1 else ".1f"
        df_temp["freq"] = df_temp["freq"] / df_temp.groupby([var])["freq"].transform("sum") if col == 2 else df_temp["freq"]
        df_temp["full vaccine"] = df_temp["full vaccine"].astype(str)

        trace = px.bar(df_temp, x = var, y = "freq", color = "full vaccine", text_auto = text_auto,
                       category_orders = {"full vaccine": ["0", "1"]},
                       color_discrete_map = {"0": px.colors.qualitative.Plotly[1], "1": px.colors.qualitative.Plotly[0]})

        for data in trace.data:
            fig.add_trace(data, row = 1, col = col)
    
    (fig
     .update_layout(barmode = "stack",
                    title = dict(text = var.title(), x = 0.5),
                    xaxis = dict(title = None, tickvals = tickvals),
                    xaxis2 = dict(title = None, tickvals = tickvals),
                    yaxis = dict(title = "Count", tickformat = ","),
                    yaxis2 = dict(title = "Percent"),
                    legend = dict(title = "Full Vaccine")
                    )
    )
    
    
    names = set()
    fig.for_each_trace(lambda trace: trace.update(showlegend = False) if (trace.name in names) else names.add(trace.name))
    fig.show()

In [ ]:
with open('fig_dat.pickle', 'wb') as file:
    pickle.dump(fig.data, file, protocol = pickle.HIGHEST_PROTOCOL)
